# PyWake-Simulation vs. ZXTM-LiDAR auf T07, mit 10-Minuten-Daten

Gleicher Ablauf wie im Notebook `02_PyWake_vs_ZXTM_REWS.ipynb`, aber mit den **10-Minuten-Dateien des ZXTM** (`Wind10_*`), in denen das Gerät selbst schon über 10 Minuten gemittelt hat.

**Unterschiede zum 1-Minuten-Notebook:**

- Turbinendaten und Simulation laufen in 10-Minuten-Schritten (`FREQ = "10min"`).
- Die LiDAR-Messwerte kommen direkt aus den 10-Minuten-Dateien, wir mitteln sie nicht selbst.
- **Zeitstempel-Prüfung:** Es ist nicht dokumentiert, ob ein 10-Minuten-Zeitstempel den Anfang oder das Ende des Intervalls bezeichnet. Das Notebook bestimmt das automatisch, indem es die 10-Minuten-Datei mit eigenen 10-Minuten-Mitteln der hochaufgelösten Daten vergleicht.
- Intervalle, in denen das Steering umgeschaltet wurde, werden entfernt.
- Die Auswahl der Höhenstreifen richtet sich nach dem Anteil der Intervalle mit Messwert.

**Gleich geblieben:** Geometrie (Bogen-Mittelpunkt-Version), Projektion auf LOS, REWS-Regel (vorhandene Streifen, mindestens 3 inklusive Nabenhöhe, Gewichte nach Variante A), Kennzahlen.

**Hinweis:** Der voreingestellte Zeitraum (4,5 Stunden) ergibt nur 27 Intervalle. Für belastbare Kennzahlen sollte man später einen längeren Zeitraum wählen (`START`, `END_EXCL`), z. B. mehrere Tage im Februar 2026. Fastlog- und LiDAR-Daten werden pro Tag zwischengespeichert, ab dem zweiten Durchlauf geht das Laden schneller.

## 1. Einstellungen

In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Transformer

from py_wake.deflection_models.jimenez import JimenezWakeDeflection
from py_wake.flow_map import HorizontalGrid, Points
from py_wake.literature.gaussian_models import Bastankhah_PorteAgel_2014
from py_wake.site import UniformSite
from py_wake.site.shear import PowerShear
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine

from hot_open.fastlog_helpers import load_hot_fl_data
from hot_open.lidar_helpers import load_zx_lidar_10min_data, load_zx_lidar_fl_data
from hot_open.settings import get_data_dir, get_filestore_dir
from hot_open.sourcing_data import ensure_hot_data_files

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

# Zeitraum
START = pd.Timestamp("2026-02-20 00:00", tz="UTC")
END_EXCL = pd.Timestamp("2026-02-20 04:30", tz="UTC")

FREQ = "10min"

# Wake-Modell und Anströmung
WAKE_K = 0.0324555
TI = 0.1
SHEAR_ALPHA = 0.2
HUB_H = 59.0
ROTOR_R = 41.0

# Korrektur der Anströmung von T01: ws_korrigiert = ws * WS_FACTOR + WS_ADD
WS_FACTOR = 1.0
WS_ADD = 0.0

# Optionaler Filter auf das Steering-Fenster T03 -> T07
APPLY_WD_FILTER = False
WD_WINDOW = (192.0, 224.0)

# ZXTM-Geometrie
RANGES_M = [85.0, 126.0, 208.0]
HEIGHTS_M = [26.2, 42.6, 59.0, 75.4, 91.8]
SLICE_HALF = 8.2
SIDES = ["Right", "Left"]                    # rechts = West, links = Ost (bei Wind aus SSW)
HALF_ANGLE_DEG = 15.0
LIDAR_Z = HUB_H + 3.2                        # Deployment-Bericht: 3.2 m über der Rotorachse

# REWS-Regel
MIN_SHARE = 0.5                              # Streifen verwenden, wenn in >= 50 % der Intervalle gemessen
MIN_HEIGHTS = 3                              # pro Intervall mindestens 3 Streifen, inkl. Nabenhöhe

# Fastlog-Tags des Dynamic-Yaw-Controllers (True North)
YAW = "computed_driver_pre_processed_yaw_direction_true_degrees"
WD = "computed_core_post_processed_consensus_wind_direction_true_degrees"
TOGGLE = "computed_driver_post_processed_toggle_state"
WS_OFFSET = "computed_core_post_processed_core_wake_steering_offset_degrees"

LOS_COL = "{side} LOS Speed (m/s) at Rotor Segment Height {h}m"
LOS_COLS = [LOS_COL.format(side=s, h=h) for s in SIDES for h in HEIGHTS_M]

c:\Users\ryan.feusi\hill-of-towie-open-source-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Windpark-Setup

In [2]:
ensure_hot_data_files(["Hill_of_Towie_turbine_metadata.csv"])
turbine_metadata = pd.read_csv(get_data_dir() / "Hill_of_Towie_turbine_metadata.csv")
names = turbine_metadata["Turbine Name"].to_list()

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32630", always_xy=True)
utm_x, utm_y = transformer.transform(turbine_metadata["Longitude"].to_numpy(),
                                     turbine_metadata["Latitude"].to_numpy())
x = utm_x - utm_x.min()
y = utm_y - utm_y.min()

i01, i03, i07 = names.index("T01"), names.index("T03"), names.index("T07")
x07, y07 = x[i07], y[i07]

swt23 = GenericWindTurbine(name="SWT-2.3-VS-82", diameter=2 * ROTOR_R, hub_height=HUB_H,
                           power_norm=2300, turbulence_intensity=TI)

INFO hot_open.sourcing_data: ensure_hot_data_files: all 1 requested files already present at C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\data (and no missing small files in cached metadata), skipping download


## 3. Turbinendaten laden und auf 10 Minuten mitteln

Wie im 1-Minuten-Notebook: True-North-Tags des Controllers, zirkuläre Mittelung der Richtungen, Gierwinkel von T03 = Windrichtung − Gondelrichtung.

Die 10-Minuten-Mittel werden mit dem **Intervallanfang** beschriftet (Voreinstellung von pandas): Der Zeitstempel 01:20 steht für 01:20–01:30.

Zusätzlich werden Intervalle entfernt, in denen der Toggle umgeschaltet hat (Anteil "an" weder ~0 noch ~1).

In [3]:
fl = load_hot_fl_data(
    data_dir=get_filestore_dir(),
    wtg_numbers=[1, 3, 7],
    start_dt=START,
    end_dt_excl=END_EXCL,
    extra_tags=[YAW, WD, TOGGLE, WS_OFFSET],
)


def circ_mean_resample(series, freq):
    rad = np.deg2rad(series)
    return np.rad2deg(np.arctan2(np.sin(rad).resample(freq).mean(),
                                 np.cos(rad).resample(freq).mean())) % 360


def circ_diff(a, b):
    return (a - b + 180) % 360 - 180


m = pd.DataFrame({
    "ws_t01":     fl[("T01", "AcWindSp_AcWindSp")].resample(FREQ).mean(),
    "wd_t01":     circ_mean_resample(fl[("T01", WD)], FREQ),
    "wd_t03":     circ_mean_resample(fl[("T03", WD)], FREQ),
    "yaw_t03":    circ_mean_resample(fl[("T03", YAW)], FREQ),
    "yaw_t07":    circ_mean_resample(fl[("T07", YAW)], FREQ),
    "offset_t03": fl[("T03", WS_OFFSET)].resample(FREQ).mean(),
    "toggle_t03": fl[("T03", TOGGLE)].resample(FREQ).mean(),
})
m["yaw_mis_t03"] = circ_diff(m["wd_t03"], m["yaw_t03"])
m = m.dropna(subset=["ws_t01", "wd_t01", "yaw_mis_t03", "yaw_t07", "toggle_t03"])

n_before = len(m)
m = m[(m["toggle_t03"] < 0.05) | (m["toggle_t03"] > 0.95)]
m["toggle_on"] = m["toggle_t03"] > 0.5
print(f"{len(m)} Intervalle ({n_before - len(m)} wegen Umschaltung entfernt), davon {m['toggle_on'].sum()} mit Steering an")
m.round(2)

INFO hot_open.fastlog_helpers: Reading: C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\cache\fl_resampled\HOT\2304510\20260220_EIlqz_Z3Vc3ki-UJiOMVKhMeDnJLhA3m.parquet
INFO hot_open.fastlog_helpers: Reading: C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\cache\fl_resampled\HOT\2304512\20260220_OXg6c4tKlhtGN-YBZ90MA9rBoixaCNpU.parquet
INFO hot_open.fastlog_helpers: Reading: C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\cache\fl_resampled\HOT\2304516\20260220_Fb8AESKTyNWUf-OuDVGMiizyMgkYjxdx.parquet


27 Intervalle (0 wegen Umschaltung entfernt), davon 13 mit Steering an


,ws_t01,wd_t01,wd_t03,yaw_t03,yaw_t07,offset_t03,toggle_t03,yaw_mis_t03,toggle_on
timestamp,,,,,,,,,
2026-02-20 00:00:00+00:00,7.70,200.31,197.55,196.10,196.04,-12.30,0.0,1.46,False
2026-02-20 00:10:00+00:00,6.77,198.66,195.76,196.48,197.39,-9.09,0.0,-0.72,False
2026-02-20 00:20:00+00:00,5.56,191.92,189.39,189.26,190.88,0.00,0.0,0.13,False
2026-02-20 00:30:00+00:00,8.38,197.89,194.59,192.69,194.62,-4.90,0.0,1.90,False
2026-02-20 00:40:00+00:00,9.80,199.97,197.01,202.31,196.19,-7.78,1.0,-5.30,True
2026-02-20 00:50:00+00:00,8.87,198.35,194.86,201.58,194.76,-6.04,1.0,-6.72,True
2026-02-20 01:00:00+00:00,8.44,201.16,197.03,205.47,195.34,-11.53,1.0,-8.44,True
2026-02-20 01:10:00+00:00,8.65,205.09,201.62,215.83,200.21,-15.00,1.0,-14.21,True
2026-02-20 01:20:00+00:00,8.37,203.88,200.89,216.62,199.41,-14.81,1.0,-15.73,True


## 4. PyWake-Simulation

Wie im 1-Minuten-Notebook, ein Fall pro 10-Minuten-Intervall. `sim_steer` mit dem gemessenen Gierwinkel von T03, `sim_ref` mit allen Turbinen im Wind.

In [4]:
site = UniformSite(p_wd=[1], ti=TI, shear=PowerShear(h_ref=HUB_H, alpha=SHEAR_ALPHA))
wf_model = Bastankhah_PorteAgel_2014(site, swt23, k=WAKE_K, deflectionModel=JimenezWakeDeflection())

time = m.index.tz_convert(None).to_numpy()
wd_in = m["wd_t01"].to_numpy()
ws_in = np.clip(m["ws_t01"].to_numpy() * WS_FACTOR + WS_ADD, 0.1, None)

yaw = np.zeros((len(x), len(m)))
yaw[i03, :] = m["yaw_mis_t03"].to_numpy()

sim_steer = wf_model(x, y, wd=wd_in, ws=ws_in, time=time, yaw=yaw, tilt=0)
sim_ref = wf_model(x, y, wd=wd_in, ws=ws_in, time=time, yaw=np.zeros_like(yaw), tilt=0)

## 5. ZXTM-Daten laden

**Beide** Varianten:

- die **10-Minuten-Datei** (`Wind10_*`) für den eigentlichen Vergleich,
- die **hochaufgelösten Daten** (`Wind_*`) nur für die Zeitstempel-Prüfung in Abschnitt 6.

In beiden werden die Platzhalterwerte 9987–9999 entfernt. Die 10-Minuten-Datei wird mit einem Intervall Reserve auf beiden Seiten geladen, damit eine eventuelle Verschiebung der Zeitstempel keine Werte am Rand abschneidet.

In [5]:
def remove_sentinels(df):
    # Laut Data Dictionary: Werte 9987 bis 9999 bedeuten ungültige Messung
    df = df.copy()
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].mask((df[num_cols] >= 9987) & (df[num_cols] <= 9999))
    return df


zxtm10 = remove_sentinels(load_zx_lidar_10min_data(
    data_dir=get_data_dir() / "lidar_data",
    lidar_unit_id="5060",
    start_dt=START - pd.Timedelta("10min"),
    end_dt_excl=END_EXCL + pd.Timedelta("10min"),
    remove_bad_values=True,
))

zxtm_hr = remove_sentinels(load_zx_lidar_fl_data(
    data_dir=get_data_dir() / "lidar_data",
    lidar_unit_id="5060",
    start_dt=START,
    end_dt_excl=END_EXCL,
    remove_bad_values=True,
))

print("10-min-Datei:", zxtm10.shape, "| hochaufgelöst:", zxtm_hr.shape)

INFO hot_open.sourcing_data: ensure_extracted: lidar_data.zip already extracted (sentinel C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\data\lidar_data\timeseries present), skipping
INFO hot_open.lidar_helpers: Reading 4 parquet files from C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\data\lidar_data\timeseries\5060
Reading 5060 10-min parquet files: 100%|██████████| 4/4 [00:00<00:00, 25.28it/s]
INFO hot_open.sourcing_data: ensure_extracted: lidar_data.zip already extracted (sentinel C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\data\lidar_data\timeseries present), skipping
INFO hot_open.lidar_helpers: Reading 1 fastlog parquet files from C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\data\lidar_data\timeseries\5060
Reading 5060 fastlog parquet files: 100%|██████████| 1/1 [00:00<00:00,  3.67it/s]
INFO hot_open.lidar_helpers: Writing: C:\Users\ryan.feusi\temp\hill-of-towie-open-source-analysis\cache\lidar_raw\5060_20260220000000_202

10-min-Datei: (28, 304) | hochaufgelöst: (15202, 37)


### Kontrolle: Ist die 10-Minuten-Datei so aufgebaut, wie erwartet?

Zu erwarten: eine Spalte `Range (m)` (eine Zeile pro Intervall und Distanz) und dieselben LOS-Spalten wie in den hochaufgelösten Daten. Falls hier eine Fehlermeldung kommt, ist die Datei anders aufgebaut, und die folgenden Zellen müssen angepasst werden.

In [6]:
missing = [c for c in ["Range (m)", *LOS_COLS] if c not in zxtm10.columns]
if missing:
    raise KeyError(f"Diese Spalten fehlen in der 10-min-Datei: {missing}")

print(zxtm10["Range (m)"].value_counts().sort_index())
print("\nWeitere Spalten mit Bezug zur Nabenhöhe oder Statistik:")
print([c for c in zxtm10.columns if ("59.0m" in c and "LOS" not in c) or "Packets" in c or "Std" in c])

KeyError: "Diese Spalten fehlen in der 10-min-Datei: ['Range (m)', 'Right LOS Speed (m/s) at Rotor Segment Height 26.2m', 'Right LOS Speed (m/s) at Rotor Segment Height 42.6m', 'Right LOS Speed (m/s) at Rotor Segment Height 59.0m', 'Right LOS Speed (m/s) at Rotor Segment Height 75.4m', 'Right LOS Speed (m/s) at Rotor Segment Height 91.8m', 'Left LOS Speed (m/s) at Rotor Segment Height 26.2m', 'Left LOS Speed (m/s) at Rotor Segment Height 42.6m', 'Left LOS Speed (m/s) at Rotor Segment Height 59.0m', 'Left LOS Speed (m/s) at Rotor Segment Height 75.4m', 'Left LOS Speed (m/s) at Rotor Segment Height 91.8m']"

In [7]:
cols = list(zxtm10.columns)
print(zxtm10.shape, "| Zeilen pro Zeitstempel (max):", zxtm10.index.value_counts().max())
print(len(cols), "Spalten\n")

for key in ["Range", "LOS", "Segment", "85", "126", "208", "Wind Speed", "Yaw", "Packets"]:
    hits = [c for c in cols if key in c]
    print(f"'{key}': {len(hits)} Spalten, z. B.:")
    for c in hits[:8]:
        print("   ", c)

pd.Series(cols).to_csv("zxtm10_spalten.csv", index=False)

(28, 304) | Zeilen pro Zeitstempel (max): 1
304 Spalten

'Range': 0 Spalten, z. B.:
'LOS': 0 Spalten, z. B.:
'Segment': 180 Spalten, z. B.:
    PDAC Packets in Average at Rotor Segment Height 26.2m at 208m
    PDAC Packets in Average at Rotor Segment Height 42.6m at 208m
    PDAC Packets in Average at Rotor Segment Height 59.0m at 208m
    PDAC Packets in Average at Rotor Segment Height 75.4m at 208m
    PDAC Packets in Average at Rotor Segment Height 91.8m at 208m
    PDAC Horizontal Wind Speed (m/s) at Rotor Segment Height 26.2m at 208m
    PDAC Horizontal Wind Speed (m/s) at Rotor Segment Height 42.6m at 208m
    PDAC Horizontal Wind Speed (m/s) at Rotor Segment Height 59.0m at 208m
'85': 48 Spalten, z. B.:
    FD Packets in Average at 85m
    FD Wind Yaw Misalignment (deg) at 85m
    FD Wind Yaw Misalignment (deg) Min at 85m
    FD Wind Yaw Misalignment (deg) Max at 85m
    FD Wind Yaw Misalignment Std. Dev. (deg) at 85m
    FD Horizontal Wind Speed (m/s) at Hub Height at 85m
    F

## 6. Zeitstempel-Prüfung: Anfang oder Ende des Intervalls?

Zu mitteln: die hochaufgelösten Daten selbst auf 10 Minuten, einmal beschriftet mit dem Intervall**anfang** (`label="left"`), einmal mit dem **Ende** (`label="right"`). Die Blöcke sind identisch, nur die Beschriftung unterscheidet sich. Die Variante mit der kleineren Abweichung zur 10-Minuten-Datei verrät die Konvention.

Weil die Turbinendaten mit dem Anfang beschriftet sind, werden die Zeitstempel der 10-Minuten-Datei gegebenenfalls um 10 Minuten zurückverschoben (`TS_SHIFT`).

Liegt keine Variante nahe an der Datei, mittelt das Gerät anders als gedacht. Dann Plot anschauen.

In [ ]:
test_col = LOS_COL.format(side="Right", h=59.0)
deviations = {}
compare = {}
for r in RANGES_M:
    own = zxtm_hr[zxtm_hr["Range (m)"] == r][test_col]
    device = zxtm10[zxtm10["Range (m)"] == r][test_col]
    cmp = pd.DataFrame({
        "10-min-Datei": device,
        "Anfang": own.resample("10min", label="left").mean(),
        "Ende": own.resample("10min", label="right").mean(),
    }).dropna()
    compare[r] = cmp
    deviations[r] = cmp[["Anfang", "Ende"]].sub(cmp["10-min-Datei"], axis=0).abs().mean()

deviations = pd.DataFrame(deviations).T
print("Mittlere Abweichung zur 10-min-Datei [m/s] (Rechter Strahl, 59 m):")
print(deviations.round(3))

convention = "Anfang" if deviations["Anfang"].mean() < deviations["Ende"].mean() else "Ende"
TS_SHIFT = pd.Timedelta(0) if convention == "Anfang" else pd.Timedelta("-10min")
print(f"\nErgebnis: Zeitstempel bezeichnen das {convention} des Intervalls, Verschiebung {TS_SHIFT}")
if deviations[convention].mean() > 0.3:
    print("WARNUNG: Auch die bessere Variante weicht deutlich ab. Das Gerät mittelt offenbar anders. Plot prüfen!")

In [ ]:
fig, axes = plt.subplots(len(RANGES_M), 1, figsize=(14, 8), sharex=True)
for ax, r in zip(axes, RANGES_M):
    compare[r].plot(ax=ax, style=["k-o", "b--", "r:"], legend=(r == RANGES_M[0]))
    ax.set_ylabel(f"{r:.0f} m\nLOS [m/s]")
    ax.grid(alpha=0.3)
axes[0].set_title("Zeitstempel-Prüfung: 10-min-Datei vs. eigene 10-min-Mittel der hochaufgelösten Daten")
plt.show()

### Welche Höhenstreifen verwenden wir pro Distanz?

Ein Streifen wird verwendet, wenn er auf **beiden** Seiten in mindestens `MIN_SHARE` der Intervalle einen Wert hat.

In [ ]:
zxtm10_win = zxtm10.copy()
zxtm10_win.index = zxtm10_win.index + TS_SHIFT
zxtm10_win = zxtm10_win[(zxtm10_win.index >= START) & (zxtm10_win.index < END_EXCL)]

coverage10 = zxtm10_win.groupby("Range (m)")[LOS_COLS].apply(lambda g: g.notna().mean())
display(coverage10.T.round(2))

REWS_HEIGHTS = {
    r: [h for h in HEIGHTS_M
        if min(coverage10.loc[r, LOS_COL.format(side=s, h=h)] for s in SIDES) >= MIN_SHARE]
    for r in RANGES_M
}
REWS_HEIGHTS

## 7. Geometrie (Bogen-Mittelpunkt-Version)

Für jede Distanz, jeden Streifen und jede Seite: Abtastpunkte gleichmäßig entlang des Messkreises verteilen, die Punkte im Streifen und auf der richtigen Seite auswählen und deren Mittelpunkt als Messposition verwenden. Grundlage ist die mittlere Gondelrichtung von T07 im Intervall.

In [ ]:
TAN_HALF = np.tan(np.deg2rad(HALF_ANGLE_DEG))


def zxtm_points_3d(yaw_deg, n_samples=720):
    yaw_r = np.deg2rad(yaw_deg)
    axis_x, axis_y = np.sin(yaw_r), np.cos(yaw_r)
    right_x, right_y = np.cos(yaw_r), -np.sin(yaw_r)
    theta = np.linspace(0, 2 * np.pi, n_samples, endpoint=False)

    pts = []
    for r in RANGES_M:
        rho = r * TAN_HALF
        slant = np.hypot(r, rho)
        circle_side = rho * np.cos(theta)
        circle_z = LIDAR_Z + rho * np.sin(theta)
        for h in HEIGHTS_M:
            in_slice = np.abs(circle_z - h) <= SLICE_HALF
            for side, on_side in [("Right", circle_side > 0), ("Left", circle_side < 0)]:
                mask = in_slice & on_side
                reachable = bool(mask.any())
                lateral = circle_side[mask].mean() if reachable else 0.0
                z_mean = circle_z[mask].mean() if reachable else h
                pts.append({
                    "range": r, "height": h, "side": side, "reachable": reachable,
                    "x": x07 + r * axis_x + lateral * right_x,
                    "y": y07 + r * axis_y + lateral * right_y,
                    "z": z_mean,
                    "c_axis": r / slant,
                    "c_side": lateral / slant,
                })
    return pts

## 8. PyWake an den Messpunkten, Projektion auf LOS, Messwerte dazu

`LOS = U · (c_axis · cos Δ + c_side · sin Δ)` mit `Δ = Windrichtung − Gondelrichtung T07`.

Die Messwerte kommen direkt aus der 10-Minuten-Datei (mit der in Abschnitt 6 bestimmten Zeitverschiebung).

In [ ]:
rows = []
for i_t, ts in enumerate(m.index):
    yaw07 = m["yaw_t07"].iloc[i_t]
    pts = zxtm_points_3d(yaw07)
    grid = Points(np.array([p["x"] for p in pts]),
                  np.array([p["y"] for p in pts]),
                  np.array([p["z"] for p in pts]))
    ws_steer = sim_steer.flow_map(grid=grid, time=time[i_t]).WS_eff.values.ravel()
    ws_ref = sim_ref.flow_map(grid=grid, time=time[i_t]).WS_eff.values.ravel()

    delta = np.deg2rad(wd_in[i_t] - yaw07)
    for p, w_steer, w_ref in zip(pts, ws_steer, ws_ref):
        proj = p["c_axis"] * np.cos(delta) + p["c_side"] * np.sin(delta)
        rows.append({"time": ts, "range": p["range"], "height": p["height"], "side": p["side"],
                     "los_steer": w_steer * proj, "los_ref": w_ref * proj})
sim_los = pd.DataFrame(rows)

meas = []
for r in RANGES_M:
    sub = zxtm10_win[zxtm10_win["Range (m)"] == r]
    for side in SIDES:
        for h in HEIGHTS_M:
            s = sub[LOS_COL.format(side=side, h=h)].reindex(m.index)
            meas.append(pd.DataFrame({"time": m.index, "range": r, "height": h,
                                      "side": side, "los_meas": s.to_numpy()}))
meas_df = pd.concat(meas)

los30 = sim_los.merge(meas_df, on=["time", "range", "height", "side"], how="left")
print(f"{los30['los_meas'].notna().mean():.0%} der Kombinationen haben einen Messwert")
los30.head()

## 9. REWS (Variante A, vorhandene Streifen pro Intervall)

`REWS = ( Σ (Aᵢ / A) · LOSᵢ³ )^(1/3)`

Pro Intervall werden die Streifen verwendet, die in diesem Intervall gemessen wurden (aus `REWS_HEIGHTS`), mindestens 3 inklusive Nabenhöhe. Die Gewichte werden nach der Logik von IEC 61400-12-1 passend zu den vorhandenen Streifen berechnet. Simulation und Messung bekommen dieselben Streifen und Gewichte.

In [ ]:
def strip_area(z_lo, z_hi, r=ROTOR_R, hub=HUB_H):
    F = lambda yy: yy * np.sqrt(r**2 - yy**2) + r**2 * np.arcsin(yy / r)
    a, b = np.clip([z_lo - hub, z_hi - hub], -r, r)
    return F(b) - F(a)


def rews_weights(heights):
    h = sorted(heights)
    bounds = [HUB_H - ROTOR_R] + [(h[i] + h[i + 1]) / 2 for i in range(len(h) - 1)] + [HUB_H + ROTOR_R]
    total = np.pi * ROTOR_R**2
    return {hh: strip_area(bounds[i], bounds[i + 1]) / total for i, hh in enumerate(h)}


def compute_rews(los_df, value_cols):
    parts = []
    for (r, side), g in los_df.groupby(["range", "side"]):
        hs_all = REWS_HEIGHTS[r]
        piv = g.pivot(index="time", columns="height", values=value_cols)
        available = piv["los_meas"][hs_all].notna()
        recs = []
        for ts, ok in available.iterrows():
            hs = [h for h in hs_all if ok[h]]
            if len(hs) < MIN_HEIGHTS or HUB_H not in hs:
                continue
            w = rews_weights(hs)
            rec = {"time": ts, "range": r, "side": side, "n_heights": len(hs)}
            for var in value_cols:
                rec[var.replace("los", "rews")] = sum(w[h] * piv.loc[ts, (var, h)] ** 3 for h in hs) ** (1 / 3)
            recs.append(rec)
        parts.append(pd.DataFrame(recs))
    return pd.concat(parts, ignore_index=True)


rews_df = compute_rews(los30, ["los_meas", "los_steer", "los_ref"])
rews_df["toggle_on"] = rews_df["time"].map(m["toggle_on"])
rews_df["wd_ok"] = rews_df["time"].map(m["wd_t01"].between(*WD_WINDOW))
rews_df.groupby(["range", "side"])["n_heights"].value_counts().unstack(fill_value=0)

## 10. Kennzahlen

Bias, RMSE und NRMSE pro Zeitreihe, Score als Mittelwert der 6 NRMSE-Werte. **Nicht direkt mit dem 1-Minuten-Notebook vergleichbar:** 10-Minuten-Mittel sind weniger verrauscht, der NRMSE fällt dadurch fast automatisch tiefer aus. Aussagekräftig ist der Vergleich "mit" gegen "ohne Steering".

In [ ]:
scored = rews_df[rews_df["wd_ok"]] if APPLY_WD_FILTER else rews_df


def metrics(df, sim_col):
    e = df[sim_col] - df["rews_meas"]
    rmse_val = np.sqrt((e**2).mean())
    return pd.Series({"n": int(e.notna().sum()), "bias": e.mean(), "rmse": rmse_val,
                      "nrmse_%": 100 * rmse_val / df["rews_meas"].mean()})


stats = pd.concat({
    "steer": scored.groupby(["range", "side"]).apply(lambda g: metrics(g, "rews_steer"), include_groups=False),
    "ref": scored.groupby(["range", "side"]).apply(lambda g: metrics(g, "rews_ref"), include_groups=False),
}, axis=1).round(3)
display(stats)

score_steer = stats[("steer", "nrmse_%")].mean()
score_ref = stats[("ref", "nrmse_%")].mean()
print(f"Score mit Steering:  {score_steer:.2f} %")
print(f"Score ohne Steering: {score_ref:.2f} %")
print(f"Skill gegenüber 'ohne Steering': {1 - score_steer / score_ref:+.1%}")

In [ ]:
stats_toggle = pd.concat({
    "steer": scored.groupby(["range", "side", "toggle_on"]).apply(lambda g: metrics(g, "rews_steer"), include_groups=False),
    "ref": scored.groupby(["range", "side", "toggle_on"]).apply(lambda g: metrics(g, "rews_ref"), include_groups=False),
}, axis=1)
stats_toggle.xs("nrmse_%", axis=1, level=1).unstack("toggle_on").round(2)

## 11. Plots

### 11.1 REWS-Zeitreihen

Bei 10-Minuten-Daten mit Punkten markiert, damit einzelne Intervalle erkennbar sind.

In [ ]:
wide = rews_df.pivot_table(index="time", columns=["range", "side"],
                           values=["rews_meas", "rews_ref", "rews_steer"])
toggle_on = m["toggle_on"].reindex(wide.index).fillna(False)

styles = [
    ("rews_meas",  dict(color="black", marker="o", ms=4),             "Messung (ZXTM, 10 min)"),
    ("rews_ref",   dict(color="tab:blue", ls="--", marker="s", ms=3), "PyWake ohne Steering"),
    ("rews_steer", dict(color="tab:red", marker="^", ms=4),           "PyWake mit WS von T03"),
]

fig, axes = plt.subplots(len(RANGES_M), 2, figsize=(16, 10), sharex=True, sharey=True)
for row, r in enumerate(RANGES_M):
    for col, side in enumerate(SIDES):
        ax = axes[row, col]
        for var, style, label in styles:
            s = wide[(var, r, side)]
            ax.plot(s.index, s, label=label, **style)
        ax.fill_between(wide.index, 0, 1, where=toggle_on, transform=ax.get_xaxis_transform(),  # type: ignore
                        color="tab:green", alpha=0.15)
        ax.set_title(f"{side} beam ({'West' if side == 'Right' else 'Ost'}), {r:.0f} m", fontsize=10)
        ax.grid(alpha=0.3)
        if col == 0:
            ax.set_ylabel("REWS (LOS) [m/s]")
axes[0, 0].legend(ncol=3, fontsize=8)
fig.suptitle("ZXTM auf T07: REWS aus LOS-Werten, Variante A, 10-Minuten-Daten")
fig.tight_layout()
plt.show()

### 11.2 NRMSE pro Zeitreihe und Streudiagramm

Links: NRMSE der 6 Zeitreihen mit und ohne Steering. Rechts: simulierte gegen gemessene REWS. Punkte auf der Diagonale bedeuten perfekte Übereinstimmung, Punkte darunter: PyWake zu tief.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

labels = [f"{r:.0f} m\n{side}" for r, side in stats.index]
xpos = np.arange(len(labels))
axes[0].bar(xpos - 0.2, stats[("ref", "nrmse_%")], width=0.4, color="tab:blue", label="ohne Steering")
axes[0].bar(xpos + 0.2, stats[("steer", "nrmse_%")], width=0.4, color="tab:red", label="mit WS von T03")
axes[0].set_xticks(xpos, labels)
axes[0].set_ylabel("NRMSE [%]")
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

range_colors = {85.0: "darkgreen", 126.0: "seagreen", 208.0: "yellowgreen"}
for r, c in range_colors.items():
    d = scored[scored["range"] == r]
    axes[1].scatter(d["rews_meas"], d["rews_steer"], color=c, edgecolor="k", s=30, label=f"{r:.0f} m")
lims = [scored[["rews_meas", "rews_steer"]].min().min() - 0.5, scored[["rews_meas", "rews_steer"]].max().max() + 0.5]
axes[1].plot(lims, lims, color="grey", lw=1)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_aspect("equal")
axes[1].set_xlabel("REWS gemessen [m/s]")
axes[1].set_ylabel("REWS PyWake mit Steering [m/s]")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.show()

### 11.3 Höhenprofile zu einem Intervall

Der Zeitstempel bezeichnet den Intervallanfang: `01:20` steht für 01:20–01:30 (enthält das Beispiel 01:25 der Referenzabbildung), `02:40` für 02:40–02:50.

In [ ]:
def plot_profiles(t_str):
    ts = pd.Timestamp(t_str, tz="UTC")
    snap = los30[los30["time"] == ts]
    if snap.empty:
        print(f"{t_str}: kein Intervall vorhanden (evtl. herausgefiltert)")
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharex=True, sharey=True)
    for ax, side in zip(axes, SIDES):
        for r, c in range_colors.items():
            d = snap[(snap["side"] == side) & (snap["range"] == r)
                     & (snap["height"].isin(REWS_HEIGHTS[r]))].sort_values("height")
            ax.plot(d["los_meas"], d["height"], "o-", color=c, label=f"{r:.0f} m gemessen")
            ax.plot(d["los_steer"], d["height"], "s--", color=c, alpha=0.7, label=f"{r:.0f} m PyWake")
        ax.set_title(f"{side} beam ({'West' if side == 'Right' else 'Ost'})")
        ax.set_xlabel("LOS [m/s]")
        ax.set_ylim(18, 100)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Höhe über Grund T07 [m]")
    axes[1].legend(fontsize=8)
    fig.suptitle(f"ZXTM-Höhenprofile {ts:%H:%M}–{ts + pd.Timedelta(FREQ):%H:%M} UTC: Messung vs. PyWake mit Steering")
    plt.show()


plot_profiles("2026-02-20 01:20")
plot_profiles("2026-02-20 02:40")

### 11.4 Draufsicht mit Messpunkten

In [ ]:
def plot_flowmap_with_points(t_str, sim=sim_steer, title="PyWake mit WS von T03"):
    ts = pd.Timestamp(t_str, tz="UTC")
    if ts not in m.index:
        print(f"{t_str}: kein Intervall vorhanden (evtl. herausgefiltert)")
        return
    i_t = m.index.get_loc(ts)
    grid = HorizontalGrid(x=np.linspace(-300, 600, 180), y=np.linspace(0, 1300, 260))
    fm = sim.flow_map(grid=grid, time=time[i_t])

    fig, ax = plt.subplots(figsize=(7, 9))
    fm.plot_wake_map(ax=ax)
    pts = pd.DataFrame(zxtm_points_3d(m["yaw_t07"].iloc[i_t]))
    for r, c in range_colors.items():
        p = pts[(pts["range"] == r) & (pts["height"].isin(REWS_HEIGHTS[r]))]
        ax.scatter(p["x"], p["y"], color=c, edgecolor="k", s=25, zorder=5, label=f"ZXTM {r:.0f} m")
    for n in ["T01", "T03", "T07"]:
        i = names.index(n)
        ax.annotate(n, (x[i], y[i]), xytext=(6, 6), textcoords="offset points", fontweight="bold")
    ax.set_xlim(-300, 600)
    ax.set_ylim(0, 1300)
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_title(f"{title}\n{ts:%H:%M} UTC (10 min), wd={wd_in[i_t]:.0f}°, T03-Yaw={m['yaw_mis_t03'].iloc[i_t]:+.1f}°")
    plt.show()


plot_flowmap_with_points("2026-02-20 01:20")
plot_flowmap_with_points("2026-02-20 02:40")

## Offene Punkte

- **Längerer Zeitraum:** 27 Intervalle sind zu wenig für belastbare Kennzahlen. `START` und `END_EXCL` auf mehrere Tage erweitern.
- **Zeitstempel-Konvention** der 10-Minuten-Dateien ist hier aus den Daten bestimmt. Bei ZX Lidars bestätigen lassen.
- **Mittelung im Gerät:** Falls die Zeitstempel-Prüfung größere Abweichungen zeigt, mittelt das Gerät anders als ein einfacher Mittelwert der hochaufgelösten Werte (z. B. mit Qualitätsfilter).
- Alle offenen Punkte aus dem 1-Minuten-Notebook gelten weiter (Geometrie, Windscherung, Veer, Anströmung, Wake-Parameter).